# Compile Pre-Fight Career Stats

Create one row per fight with each fighter's career statistics before that fight.

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd


DATA_DIR = Path.cwd()
if not (DATA_DIR / "all_fights.csv").exists():
    DATA_DIR = Path("data_clean")

input_path = DATA_DIR / "all_fights.csv"
output_path = DATA_DIR / "pre_fight_career_stats.csv"

raw_fights = pd.read_csv(input_path)
raw_fights["fight_id"] = np.arange(len(raw_fights))
raw_fights["date"] = pd.to_datetime(raw_fights["date"], errors="coerce")

print("Loaded:", input_path)
print("Shape:", raw_fights.shape)
raw_fights.head()

Loaded: e:\Libraries\Documents\GitHub\UFC-Predictions\data_clean\all_fights.csv
Shape: (8705, 65)


,date,winner,R_fighter,B_fighter,R_DOB,B_DOB,R_Height,B_Height,R_Reach,B_Reach,...,B_BODY_att,B_LEG_landed,B_LEG_att,B_DISTANCE_landed,B_DISTANCE_att,B_CLINCH_landed,B_CLINCH_att,B_GROUND_landed,B_GROUND_att,fight_id
0,2026-06-06,Gabriel Bonfim,Belal Muhammad,Gabriel Bonfim,"Jul 09, 1988","Aug 20, 1997","5' 11""","6' 1""","72""","72""",...,11,34,37,126,258,2,2,0,0,0
1,2026-06-06,Tom Nolan,Fares Ziam,Tom Nolan,"Mar 21, 1997","Mar 08, 2000","6' 1""","6' 3""","75""","73""",...,16,7,12,13,32,7,13,11,21,1
2,2026-06-06,Bryce Mitchell,Bryce Mitchell,Santiago Luna,"Oct 04, 1994","Aug 06, 2004","5' 10""","5' 9""","70""","74""",...,11,1,1,9,25,0,0,0,0,2
3,2026-06-06,Brendan Allen,Brendan Allen,Edmen Shahbazyan,"Dec 28, 1995","Nov 20, 1997","6' 2""","6' 2""","75""","74""",...,4,3,5,85,173,5,7,0,0,3
4,2026-06-06,Marcus McGhee,Marcus McGhee,John Yannis,"May 07, 1990","Jul 07, 1994","5' 8""","5' 7""","69""","70""",...,30,8,8,54,114,1,1,0,0,4


In [12]:
# Split fight-level rows into fighter-level rows, keeping opponent stats.

shared_cols = [
    "fight_id",
    "date",
    "winner",
    "win_by",
    "last_round",
    "last_round_time",
    "format",
]

stat_cols = [
    "KD",
    "SIG_STR_landed",
    "SIG_STR_att",
    "SIG_STR_pct",
    "TOTAL_STR_landed",
    "TOTAL_STR_att",
    "TD_landed",
    "TD_att",
    "TD_pct",
    "SUB_ATT",
    "REV",
    "CTRL",
    "HEAD_landed",
    "HEAD_att",
    "BODY_landed",
    "BODY_att",
    "LEG_landed",
    "LEG_att",
    "DISTANCE_landed",
    "DISTANCE_att",
    "CLINCH_landed",
    "CLINCH_att",
    "GROUND_landed",
    "GROUND_att",
]

opp_stat_cols = [
    "SIG_STR_landed",
    "SIG_STR_att",
    "TOTAL_STR_landed",
    "TOTAL_STR_att",
    "TD_landed",
    "TD_att",
    "HEAD_landed",
    "HEAD_att",
    "BODY_landed",
    "BODY_att",
    "LEG_landed",
    "LEG_att",
    "DISTANCE_landed",
    "DISTANCE_att",
    "CLINCH_landed",
    "CLINCH_att",
    "GROUND_landed",
    "GROUND_att",
]


def opp_name(col):
    if col.endswith("_landed"):
        return col.replace("_landed", "_opp_landed")
    if col.endswith("_att"):
        return col.replace("_att", "_opp_att")
    return f"{col}_opp"


r_rows = raw_fights[
    ["R_fighter", "B_fighter"]
    + shared_cols
    + [f"R_{col}" for col in stat_cols]
    + [f"B_{col}" for col in opp_stat_cols]
].copy()

r_rows = r_rows.rename(columns={
    "R_fighter": "fighter_name",
    "B_fighter": "opponent",
    **{f"R_{col}": col for col in stat_cols},
    **{f"B_{col}": opp_name(col) for col in opp_stat_cols},
})

b_rows = raw_fights[
    ["B_fighter", "R_fighter"]
    + shared_cols
    + [f"B_{col}" for col in stat_cols]
    + [f"R_{col}" for col in opp_stat_cols]
].copy()

b_rows = b_rows.rename(columns={
    "B_fighter": "fighter_name",
    "R_fighter": "opponent",
    **{f"B_{col}": col for col in stat_cols},
    **{f"R_{col}": opp_name(col) for col in opp_stat_cols},
})

fighter_fights = pd.concat([r_rows, b_rows], ignore_index=True)

fighter_fights["winner"] = fighter_fights["winner"].fillna("").astype(str).str.strip()
fighter_fights["result"] = np.where(
    fighter_fights["winner"].eq(""),
    "Draw",
    np.where(fighter_fights["fighter_name"].eq(fighter_fights["winner"]), "Win", "Loss"),
)

print("Fighter-fight shape:", fighter_fights.shape)
fighter_fights.head()

Fighter-fight shape: (17410, 52)


,fighter_name,opponent,fight_id,date,winner,win_by,last_round,last_round_time,format,KD,...,BODY_opp_att,LEG_opp_landed,LEG_opp_att,DISTANCE_opp_landed,DISTANCE_opp_att,CLINCH_opp_landed,CLINCH_opp_att,GROUND_opp_landed,GROUND_opp_att,result
0,Belal Muhammad,Gabriel Bonfim,0,2026-06-06,Gabriel Bonfim,Decision - Unanimous,5,5:00,5 Rnd (5-5-5-5-5),0,...,11,34,37,126,258,2,2,0,0,Loss
1,Fares Ziam,Tom Nolan,1,2026-06-06,Tom Nolan,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),0,...,16,7,12,13,32,7,13,11,21,Loss
2,Bryce Mitchell,Santiago Luna,2,2026-06-06,Bryce Mitchell,Submission,3,4:52,3 Rnd (5-5-5),0,...,11,1,1,9,25,0,0,0,0,Win
3,Brendan Allen,Edmen Shahbazyan,3,2026-06-06,Brendan Allen,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),0,...,4,3,5,85,173,5,7,0,0,Win
4,Marcus McGhee,John Yannis,4,2026-06-06,Marcus McGhee,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),0,...,30,8,8,54,114,1,1,0,0,Win


In [13]:
# Clean fighter-level fight data before cumulative aggregation.

win_by_map = {
    "KO/TKO": "KO_TKO",
    "TKO - Doctor's Stoppage": "KO_TKO",
    "Submission": "Submission",
    "Decision - Unanimous": "Decision",
    "Decision - Split": "Decision",
    "Decision - Majority": "Decision",
}

fighter_fights["win_by"] = (
    fighter_fights["win_by"]
    .fillna("")
    .astype(str)
    .str.strip()
    .map(win_by_map)
    .fillna("Other")
)


def time_to_seconds(x):
    if pd.isna(x):
        return 0

    x = str(x).strip()
    if x == "":
        return 0

    parts = x.split(":")
    if len(parts) == 2:
        try:
            return int(parts[0]) * 60 + int(parts[1])
        except ValueError:
            return 0

    return 0


def format_to_total_scheduled_time(x):
    if pd.isna(x):
        return 0

    x = str(x).strip()
    if x == "No Time Limit":
        return 30 * 60

    if "(" not in x or ")" not in x:
        return 0

    try:
        inside_parentheses = x.split("(")[1].split(")")[0]
        round_lengths = inside_parentheses.split("-")
        return sum(int(length) for length in round_lengths) * 60
    except ValueError:
        return 0


fighter_fights["last_round"] = pd.to_numeric(fighter_fights["last_round"], errors="coerce").fillna(0)
fighter_fights["last_round_time"] = fighter_fights["last_round_time"].apply(time_to_seconds)
fighter_fights["total_scheduled_time"] = fighter_fights["format"].apply(format_to_total_scheduled_time)
fighter_fights["CTRL"] = fighter_fights["CTRL"].apply(time_to_seconds)

fighter_fights["total_fight_time"] = (
    (fighter_fights["last_round"] - 1) * 5 * 60
    + fighter_fights["last_round_time"]
).clip(lower=0)

win_by_clean = {
    "Decision": "DEC",
    "KO_TKO": "KO_TKO",
    "Submission": "SUB",
}

fighter_fights["win_by_clean"] = fighter_fights["win_by"].map(win_by_clean).fillna("Other")
fighter_fights["is_win"] = (fighter_fights["result"] == "Win").astype(int)
fighter_fights["is_loss"] = (fighter_fights["result"] == "Loss").astype(int)
fighter_fights["win_DEC"] = ((fighter_fights["result"] == "Win") & (fighter_fights["win_by_clean"] == "DEC")).astype(int)
fighter_fights["win_KO_TKO"] = ((fighter_fights["result"] == "Win") & (fighter_fights["win_by_clean"] == "KO_TKO")).astype(int)
fighter_fights["win_SUB"] = ((fighter_fights["result"] == "Win") & (fighter_fights["win_by_clean"] == "SUB")).astype(int)
fighter_fights["loss_DEC"] = ((fighter_fights["result"] == "Loss") & (fighter_fights["win_by_clean"] == "DEC")).astype(int)
fighter_fights["loss_KO_TKO"] = ((fighter_fights["result"] == "Loss") & (fighter_fights["win_by_clean"] == "KO_TKO")).astype(int)
fighter_fights["loss_SUB"] = ((fighter_fights["result"] == "Loss") & (fighter_fights["win_by_clean"] == "SUB")).astype(int)

pct_cols_to_drop = ["SIG_STR_pct", "TD_pct"]
fighter_fights = fighter_fights.drop(columns=pct_cols_to_drop, errors="ignore")

fighter_fights.head()

,fighter_name,opponent,fight_id,date,winner,win_by,last_round,last_round_time,format,KD,...,total_fight_time,win_by_clean,is_win,is_loss,win_DEC,win_KO_TKO,win_SUB,loss_DEC,loss_KO_TKO,loss_SUB
0,Belal Muhammad,Gabriel Bonfim,0,2026-06-06,Gabriel Bonfim,Decision,5,300,5 Rnd (5-5-5-5-5),0,...,1500,DEC,0,1,0,0,0,1,0,0
1,Fares Ziam,Tom Nolan,1,2026-06-06,Tom Nolan,Decision,3,300,3 Rnd (5-5-5),0,...,900,DEC,0,1,0,0,0,1,0,0
2,Bryce Mitchell,Santiago Luna,2,2026-06-06,Bryce Mitchell,Submission,3,292,3 Rnd (5-5-5),0,...,892,SUB,1,0,0,0,1,0,0,0
3,Brendan Allen,Edmen Shahbazyan,3,2026-06-06,Brendan Allen,Decision,3,300,3 Rnd (5-5-5),0,...,900,DEC,1,0,1,0,0,0,0,0
4,Marcus McGhee,John Yannis,4,2026-06-06,Marcus McGhee,Decision,3,300,3 Rnd (5-5-5),0,...,900,DEC,1,0,1,0,0,0,0,0


In [14]:
# Calculate shifted cumulative totals, then turn them into the same career-stat features.

def safe_divide(num, denom):
    return np.where(denom > 0, num / denom, 0)


def safe_divide_nan(num, denom):
    return np.where(denom > 0, num / denom, np.nan)


numeric_cols = [
    "total_fight_time",
    "total_scheduled_time",
    "CTRL",
    "is_win",
    "is_loss",
    "win_DEC",
    "win_KO_TKO",
    "win_SUB",
    "loss_DEC",
    "loss_KO_TKO",
    "loss_SUB",
    "KD",
    "SUB_ATT",
    "REV",
    "SIG_STR_landed",
    "SIG_STR_att",
    "TOTAL_STR_landed",
    "TOTAL_STR_att",
    "TD_landed",
    "TD_att",
    "HEAD_landed",
    "HEAD_att",
    "BODY_landed",
    "BODY_att",
    "LEG_landed",
    "LEG_att",
    "DISTANCE_landed",
    "DISTANCE_att",
    "CLINCH_landed",
    "CLINCH_att",
    "GROUND_landed",
    "GROUND_att",
    "SIG_STR_opp_landed",
    "SIG_STR_opp_att",
    "TOTAL_STR_opp_landed",
    "TOTAL_STR_opp_att",
    "TD_opp_landed",
    "TD_opp_att",
    "HEAD_opp_landed",
    "HEAD_opp_att",
    "BODY_opp_landed",
    "BODY_opp_att",
    "LEG_opp_landed",
    "LEG_opp_att",
    "DISTANCE_opp_landed",
    "DISTANCE_opp_att",
    "CLINCH_opp_landed",
    "CLINCH_opp_att",
    "GROUND_opp_landed",
    "GROUND_opp_att",
]

for col in numeric_cols:
    fighter_fights[col] = pd.to_numeric(fighter_fights[col], errors="coerce").fillna(0)

fighter_fights = fighter_fights.sort_values(
    by=["fighter_name", "date", "fight_id"],
    ascending=[True, True, True],
).reset_index(drop=True)

prior_totals = fighter_fights.groupby("fighter_name")[numeric_cols].cumsum()
prior_totals = prior_totals.groupby(fighter_fights["fighter_name"]).shift(1).fillna(0)

prior_stats = fighter_fights[["fight_id", "fighter_name", "opponent", "date"]].copy()
prior_stats["total_fights"] = fighter_fights.groupby("fighter_name").cumcount()
prior_stats["total_fight_time"] = prior_totals["total_fight_time"]

total_minutes = prior_totals["total_fight_time"] / 60

prior_stats["fight_quickness"] = safe_divide(
    prior_totals["total_fight_time"],
    prior_totals["total_scheduled_time"],
)

prior_stats["win_DEC_pct"] = safe_divide(prior_totals["win_DEC"], prior_totals["is_win"])
prior_stats["win_KO_TKO_pct"] = safe_divide(prior_totals["win_KO_TKO"], prior_totals["is_win"])
prior_stats["win_SUB_pct"] = safe_divide(prior_totals["win_SUB"], prior_totals["is_win"])
prior_stats["loss_DEC_pct"] = safe_divide(prior_totals["loss_DEC"], prior_totals["is_loss"])
prior_stats["loss_KO_TKO_pct"] = safe_divide(prior_totals["loss_KO_TKO"], prior_totals["is_loss"])
prior_stats["loss_SUB_pct"] = safe_divide(prior_totals["loss_SUB"], prior_totals["is_loss"])

prior_stats["TOTAL_STR_acc"] = safe_divide(prior_totals["TOTAL_STR_landed"], prior_totals["TOTAL_STR_att"])
prior_stats["SIG_STR_acc"] = safe_divide(prior_totals["SIG_STR_landed"], prior_totals["SIG_STR_att"])
prior_stats["TD_acc"] = safe_divide(prior_totals["TD_landed"], prior_totals["TD_att"])
prior_stats["CTRL_pct"] = safe_divide(prior_totals["CTRL"], prior_totals["total_fight_time"])

prior_stats["KD_per_min"] = safe_divide(prior_totals["KD"], total_minutes)
prior_stats["SUB_ATT_per_min"] = safe_divide(prior_totals["SUB_ATT"], total_minutes)
prior_stats["REV_per_min"] = safe_divide(prior_totals["REV"], total_minutes)

prior_stats["SIG_STR_land_per_min"] = safe_divide(prior_totals["SIG_STR_landed"], total_minutes)
prior_stats["SIG_STR_att_per_min"] = safe_divide(prior_totals["SIG_STR_att"], total_minutes)
prior_stats["SIG_STR_absorbed_per_min"] = safe_divide(prior_totals["SIG_STR_opp_landed"], total_minutes)
prior_stats["SIG_STR_opp_att_per_min"] = safe_divide(prior_totals["SIG_STR_opp_att"], total_minutes)
prior_stats["SIG_STR_def"] = 1 - safe_divide(prior_totals["SIG_STR_opp_landed"], prior_totals["SIG_STR_opp_att"])

prior_stats["TOTAL_STR_land_per_min"] = safe_divide(prior_totals["TOTAL_STR_landed"], total_minutes)
prior_stats["TOTAL_STR_att_per_min"] = safe_divide(prior_totals["TOTAL_STR_att"], total_minutes)
prior_stats["TOTAL_STR_absorbed_per_min"] = safe_divide(prior_totals["TOTAL_STR_opp_landed"], total_minutes)
prior_stats["TOTAL_STR_opp_att_per_min"] = safe_divide(prior_totals["TOTAL_STR_opp_att"], total_minutes)
prior_stats["TOTAL_STR_def"] = 1 - safe_divide(prior_totals["TOTAL_STR_opp_landed"], prior_totals["TOTAL_STR_opp_att"])

prior_stats["TD_land_per_min"] = safe_divide(prior_totals["TD_landed"], total_minutes)
prior_stats["TD_att_per_min"] = safe_divide(prior_totals["TD_att"], total_minutes)
prior_stats["TD_absorbed_per_min"] = safe_divide(prior_totals["TD_opp_landed"], total_minutes)
prior_stats["TD_opp_att_per_min"] = safe_divide(prior_totals["TD_opp_att"], total_minutes)
prior_stats["TD_def"] = 1 - safe_divide_nan(prior_totals["TD_opp_landed"], prior_totals["TD_opp_att"])

for prefix in ["HEAD", "BODY", "LEG", "DISTANCE", "CLINCH", "GROUND"]:
    prior_stats[f"{prefix}_land_per_min"] = safe_divide(prior_totals[f"{prefix}_landed"], total_minutes)
    prior_stats[f"{prefix}_att_per_min"] = safe_divide(prior_totals[f"{prefix}_att"], total_minutes)
    prior_stats[f"{prefix}_absorbed_per_min"] = safe_divide(prior_totals[f"{prefix}_opp_landed"], total_minutes)
    prior_stats[f"{prefix}_opp_att_per_min"] = safe_divide(prior_totals[f"{prefix}_opp_att"], total_minutes)

prior_stats.head()

,fight_id,fighter_name,opponent,date,total_fights,total_fight_time,fight_quickness,win_DEC_pct,win_KO_TKO_pct,win_SUB_pct,...,DISTANCE_absorbed_per_min,DISTANCE_opp_att_per_min,CLINCH_land_per_min,CLINCH_att_per_min,CLINCH_absorbed_per_min,CLINCH_opp_att_per_min,GROUND_land_per_min,GROUND_att_per_min,GROUND_absorbed_per_min,GROUND_opp_att_per_min
0,1192,AJ Cunningham,Ludovit Klein,2024-03-02,0,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
1,652,AJ Cunningham,SuYoung You,2025-03-15,1,276.0,0.306667,0.0,0.0,0.0,...,11.739130,18.260870,0.434783,0.652174,0.434783,0.434783,0.0,0.0,0.869565,1.086957
2,2260,AJ Dobson,Jacob Malkoun,2022-02-12,0,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
3,1883,AJ Dobson,Armen Petrosyan,2022-10-22,1,900.0,1.000000,0.0,0.0,0.0,...,1.933333,3.800000,0.333333,0.600000,0.266667,0.466667,0.2,0.2,3.133333,4.266667
4,1451,AJ Dobson,Tafon Nchukwi,2023-08-12,2,1800.0,1.000000,0.0,0.0,0.0,...,4.500000,9.666667,0.433333,0.700000,0.533333,0.666667,0.1,0.1,1.566667,2.133333


In [15]:
# Merge pre-fight stats back to one row per fight as R_* and B_* features.

career_stat_cols = [
    "total_fights",
    "total_fight_time",
    "fight_quickness",
    "win_DEC_pct",
    "win_KO_TKO_pct",
    "win_SUB_pct",
    "loss_DEC_pct",
    "loss_KO_TKO_pct",
    "loss_SUB_pct",
    "TOTAL_STR_acc",
    "SIG_STR_acc",
    "TD_acc",
    "CTRL_pct",
    "KD_per_min",
    "SUB_ATT_per_min",
    "REV_per_min",
    "SIG_STR_land_per_min",
    "SIG_STR_att_per_min",
    "SIG_STR_absorbed_per_min",
    "SIG_STR_opp_att_per_min",
    "SIG_STR_def",
    "TOTAL_STR_land_per_min",
    "TOTAL_STR_att_per_min",
    "TOTAL_STR_absorbed_per_min",
    "TOTAL_STR_opp_att_per_min",
    "TOTAL_STR_def",
    "TD_land_per_min",
    "TD_att_per_min",
    "TD_absorbed_per_min",
    "TD_opp_att_per_min",
    "TD_def",
    "HEAD_land_per_min",
    "HEAD_att_per_min",
    "HEAD_absorbed_per_min",
    "HEAD_opp_att_per_min",
    "BODY_land_per_min",
    "BODY_att_per_min",
    "BODY_absorbed_per_min",
    "BODY_opp_att_per_min",
    "LEG_land_per_min",
    "LEG_att_per_min",
    "LEG_absorbed_per_min",
    "LEG_opp_att_per_min",
    "DISTANCE_land_per_min",
    "DISTANCE_att_per_min",
    "DISTANCE_absorbed_per_min",
    "DISTANCE_opp_att_per_min",
    "CLINCH_land_per_min",
    "CLINCH_att_per_min",
    "CLINCH_absorbed_per_min",
    "CLINCH_opp_att_per_min",
    "GROUND_land_per_min",
    "GROUND_att_per_min",
    "GROUND_absorbed_per_min",
    "GROUND_opp_att_per_min",
]

reference_path = DATA_DIR / "fighter_career_stats.csv"
if reference_path.exists():
    reference_stat_cols = [
        col for col in pd.read_csv(reference_path, nrows=0).columns.tolist()
        if col != "fighter"
    ]
    assert career_stat_cols == reference_stat_cols, "Career stat columns do not match fighter_career_stats.csv"

fight_rows = raw_fights[["fight_id", "date", "R_fighter", "B_fighter", "R_DOB", "B_DOB", "winner", "win_by"]].copy()

def calculate_age_at_fight(fight_date, dob):
    dob = pd.to_datetime(dob, errors="coerce")
    if pd.isna(fight_date) or pd.isna(dob):
        return np.nan
    return (fight_date - dob).days / 365.25


fight_rows["R_age"] = fight_rows.apply(
    lambda row: calculate_age_at_fight(row["date"], row["R_DOB"]),
    axis=1,
)
fight_rows["B_age"] = fight_rows.apply(
    lambda row: calculate_age_at_fight(row["date"], row["B_DOB"]),
    axis=1,
)
fight_rows["R_opponent"] = fight_rows["B_fighter"]
fight_rows["B_opponent"] = fight_rows["R_fighter"]
fight_rows["winner"] = fight_rows["winner"].fillna("").astype(str).str.strip()
fight_rows["winner_side"] = np.select(
    [
        fight_rows["winner"].eq(fight_rows["R_fighter"]),
        fight_rows["winner"].eq(fight_rows["B_fighter"]),
    ],
    ["R", "B"],
    default="Draw",
)
fight_rows["R_result"] = np.select(
    [fight_rows["winner_side"].eq("R"), fight_rows["winner_side"].eq("B")],
    ["Win", "Loss"],
    default="Draw",
)
fight_rows["B_result"] = np.select(
    [fight_rows["winner_side"].eq("B"), fight_rows["winner_side"].eq("R")],
    ["Win", "Loss"],
    default="Draw",
)
fight_rows["R_won"] = fight_rows["winner_side"].eq("R").astype(int)
fight_rows["B_won"] = fight_rows["winner_side"].eq("B").astype(int)

# Pre-fight Elo ratings. Ratings start at 1500 and update after each fight.
ELO_START = 1500.0
ELO_K = 40.0


def expected_elo_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def winner_side_to_scores(winner_side):
    if winner_side == "R":
        return 1.0, 0.0
    if winner_side == "B":
        return 0.0, 1.0
    return 0.5, 0.5


def calculate_prefight_elos(fights, start_rating=ELO_START, k_factor=ELO_K):
    ratings = {}
    elo_rows = []

    ordered_fights = fights.sort_values(["date", "fight_id"]).reset_index(drop=True)

    for _, row in ordered_fights.iterrows():
        r_fighter = row["R_fighter"]
        b_fighter = row["B_fighter"]
        r_elo = ratings.get(r_fighter, start_rating)
        b_elo = ratings.get(b_fighter, start_rating)

        elo_rows.append({
            "fight_id": row["fight_id"],
            "R_elo": r_elo,
            "B_elo": b_elo,
        })

        r_score, b_score = winner_side_to_scores(row["winner_side"])
        r_expected = expected_elo_score(r_elo, b_elo)
        b_expected = 1 - r_expected

        ratings[r_fighter] = r_elo + k_factor * (r_score - r_expected)
        ratings[b_fighter] = b_elo + k_factor * (b_score - b_expected)

    return pd.DataFrame(elo_rows)


fight_elos = calculate_prefight_elos(fight_rows)
fight_rows = fight_rows.merge(fight_elos, on="fight_id", how="left")

prior_merge_cols = ["fight_id", "fighter_name"] + career_stat_cols

r_prior = prior_stats[prior_merge_cols].rename(columns={
    "fighter_name": "R_fighter",
    **{col: f"R_{col}" for col in career_stat_cols},
})

b_prior = prior_stats[prior_merge_cols].rename(columns={
    "fighter_name": "B_fighter",
    **{col: f"B_{col}" for col in career_stat_cols},
})

compiled_stats = fight_rows.merge(r_prior, on=["fight_id", "R_fighter"], how="left")
compiled_stats = compiled_stats.merge(b_prior, on=["fight_id", "B_fighter"], how="left")

compiled_stats = compiled_stats.sort_values(["date", "fight_id"]).reset_index(drop=True)
compiled_stats = compiled_stats[
    (compiled_stats["R_total_fights"] >= 4)
    & (compiled_stats["B_total_fights"] >= 4)
].reset_index(drop=True)
compiled_stats["date"] = compiled_stats["date"].dt.strftime("%Y-%m-%d")

compiled_stats = compiled_stats[
    (compiled_stats["R_result"] != "Draw") &
    (compiled_stats["B_result"] != "Draw")
].reset_index(drop=True)

final_cols = [
    "fight_id",
    "date",
    "R_fighter",
    "B_fighter",
    "R_age",
    "B_age",
    "winner",
    "winner_side",
    "win_by",
    "R_elo",
    "B_elo",
] + [f"R_{col}" for col in career_stat_cols] + [f"B_{col}" for col in career_stat_cols]

compiled_stats = compiled_stats[final_cols]
compiled_stats.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", compiled_stats.shape)
compiled_stats.head()

Saved: e:\Libraries\Documents\GitHub\UFC-Predictions\data_clean\pre_fight_career_stats.csv
Shape: (3040, 121)


,fight_id,date,R_fighter,B_fighter,R_age,B_age,winner,winner_side,win_by,R_elo,...,B_DISTANCE_absorbed_per_min,B_DISTANCE_opp_att_per_min,B_CLINCH_land_per_min,B_CLINCH_att_per_min,B_CLINCH_absorbed_per_min,B_CLINCH_opp_att_per_min,B_GROUND_land_per_min,B_GROUND_att_per_min,B_GROUND_absorbed_per_min,B_GROUND_opp_att_per_min
0,8644,1995-12-16,Dan Severn,Oleg Taktarov,37.522245,28.306639,Dan Severn,R,Decision - Unanimous,1573.348793,...,0.581583,2.132472,0.038772,0.135703,0.213247,0.465267,0.000000,0.000000,0.678514,1.124394
1,8645,1995-12-16,Oleg Taktarov,Dave Beneteau,28.306639,NaN,Oleg Taktarov,R,Submission,1558.827392,...,0.000000,0.000000,0.920716,2.455243,0.153453,0.613811,3.222506,5.217391,0.153453,0.153453
2,8647,1995-12-16,Dan Severn,Paul Varelans,37.522245,26.245038,Dan Severn,R,Submission,1593.683396,...,1.912568,3.606557,0.491803,0.765027,0.327869,1.366120,0.327869,0.327869,0.928962,1.475410
3,8628,1996-05-17,Dan Severn,Ken Shamrock,37.941136,32.262834,Dan Severn,R,Decision - Split,1609.448071,...,0.205781,0.911318,0.235179,0.293974,0.146987,0.235179,1.028907,1.499265,0.146987,0.205781
4,8606,1996-12-07,Ken Shamrock,Brian Johnston,32.821355,28.361396,Ken Shamrock,R,Submission,1562.685802,...,0.203390,0.508475,1.220339,1.830508,0.508475,0.711864,0.711864,1.118644,1.627119,1.830508
